In [ ]:
#! pip install notion_client
#! pip install psycopg2
#! pip install sentence-transformers
#! pip install chromadb

In [ ]:
! pip install chromadb

In [10]:
from notion_client import Client
import psycopg2
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv
import os 

load_dotenv()

True

In [ ]:
notion = Client(auth=os.getenv("NOTION_API_KEY"))
database_id = os.getenv("NOTION_DATABASE_ID")

# 블록에서 텍스트 추출
def get_text_from_block(block):
    block_type = block["type"]
    texts = []

    # 대부분의 블록이 rich_text를 가짐
    if "rich_text" in block[block_type]:
        texts.extend([t["plain_text"] for t in block[block_type]["rich_text"]])

    # heading, bulleted_list, numbered_list 같은 경우도 포함
    if block_type in ["heading_1", "heading_2", "heading_3"]:
        texts = ["# " * (int(block_type[-1])) + "".join(texts)]
    elif block_type in ["bulleted_list_item", "numbered_list_item"]:
        texts = ["- " + "".join(texts)]
    elif block_type == "to_do":
        checked = "✅" if block[block_type]["checked"] else "⬜"
        texts = [f"{checked} " + "".join(texts)]

    return "\n".join(texts)

# 재귀적으로 모든 블록 가져오기
def get_all_blocks(block_id, depth=0):
    blocks = notion.blocks.children.list(block_id=block_id)["results"]
    content_list = []
    for block in blocks:
        content_list.append(get_text_from_block(block))

        # 자식 블록이 있으면 재귀 호출
        if block.get("has_children"):
            child_content = get_all_blocks(block["id"], depth+1)
            content_list.extend(child_content)

    return content_list


# DB에서 데이터 불러오기
results = notion.databases.query(database_id=database_id)["results"]

for page in results:
    # 제목
    title = ""
    if page["properties"]["이름"]["title"]:
        title = page["properties"]["이름"]["title"][0]["plain_text"]

    # 날짜
    date = page["properties"]["날짜"]["date"]
    if date:
        date = date["start"]
    else:
        date = None

    # 본문 (재귀적으로 가져오기)
    page_id = page["id"]
    contents = get_all_blocks(page_id)

    print("제목:", title)
    print("날짜:", date)
    print("내용:", "\n".join(c for c in contents if c.strip()) or "(내용 없음)")
    print("=" * 80)